# Time to DCI

- time to DCI
- time to DCI related infarction
- time to CVS (as reported in the SOS registry)
- time to CT (as extracted from PDMS)

Possibly tie this with new definition of DCI as ischemia and not infarction?

In [ ]:
import pandas as pd
from matplotlib.pyplot import viridis

from utils.utils import load_encrypted_xlsx
from utils.utils import safe_conversion_to_datetime
from scipy.stats import mannwhitneyu
import seaborn as sns
from matplotlib import pyplot as plt

Load data

In [ ]:
post_hoc_corrected_registry_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx'
outcome_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/sos_sah_data/follow_up/aSAH_DATA_2009_2024_18122024.xlsx'
foch_registry_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/foch_data/HSA_dci_manual_extraction.xlsx'
ct_timings_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/pdms_data/extracted_data/20240207_SAH_SOS_CT.csv'
registry_pdms_correspondence_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/pdms_data/registry_pdms_correspondence.csv'

In [ ]:
# verify all paths exist
import os
paths = [post_hoc_corrected_registry_path, outcome_data_path, foch_registry_data_path, ct_timings_path, registry_pdms_correspondence_path]
for path in paths:
    if not os.path.exists(path):
        print(f"Path does not exist: {path}")

In [ ]:
registry_df = load_encrypted_xlsx(post_hoc_corrected_registry_path)
outcome_df = load_encrypted_xlsx(outcome_data_path)
ct_timings_df = pd.read_csv(ct_timings_path, sep=';', decimal='.')
registry_pdms_correspondence_df = pd.read_csv(registry_pdms_correspondence_path)
registry_pdms_correspondence_df['Date_birth'] = pd.to_datetime(registry_pdms_correspondence_df['Date_birth'], format='%d.%m.%Y')

Preprocessing

In [ ]:

# for patients with Date_CVS_Start nan but Date_CVS_DSA not nan, set Date_CVS_Start = Date_CVS_DSA
registry_df.loc[(registry_df['Date_CVS_Start'].isnull()) & (
    registry_df['Date_CVS_DSA'].notnull()), 'Date_CVS_Start'] = registry_df['Date_CVS_DSA']
# for patients with Date_CVS_Start nan but Date_CVS_CTA not nan, set Date_CVS_Start = Date_CVS_CTA
registry_df.loc[(registry_df['Date_CVS_Start'].isnull()) & (
    registry_df['Date_CVS_CTA'].notnull()), 'Date_CVS_Start'] = registry_df['Date_CVS_CTA']
# for patients with Date_CVS_Start nan but Date_CVS_TCD not nan, set Date_CVS_Start = Date_CVS_TCD
registry_df.loc[(registry_df['Date_CVS_Start'].isnull()) & (
    registry_df['Date_CVS_TCD'].notnull()), 'Date_CVS_Start'] = registry_df['Date_CVS_TCD']

# patients with Date_CVS_Start not na but with  but CVS_YN = 0, in this case we should set CVS_YN = 1
registry_df.loc[(registry_df['CVS_YN'] == 0) & (
    registry_df['Date_CVS_Start'].apply(safe_conversion_to_datetime).notnull()), 'CVS_YN'] = 1

In [ ]:
# if Date_Ictus is nan, set it to Date_admission
registry_df.loc[registry_df['Date_Ictus'].isnull(), 'Date_Ictus'] = registry_df['Date_admission']

In [ ]:
ct_timings_df = ct_timings_df.merge(registry_pdms_correspondence_df, on='pNr', how='left')
ct_timings_df.rename(columns={'JoinedName': 'Name'}, inplace=True)
ct_timings_df = ct_timings_df.merge(registry_df[['SOS-CENTER-YEAR-NO.','Name', 'Date_birth', 'Date_admission', 'Date_Ictus', 'CVS_YN']], on=['SOS-CENTER-YEAR-NO.','Name', 'Date_birth'], how='left')

In [ ]:
# for each image check if it is the first image of DCI ischemia
registry_df['ct_date'] = registry_df['Date_DCI_ischemia_first_image'].apply(safe_conversion_to_datetime)
ct_timings_df['ct_date'] = ct_timings_df['timeAktion'].apply(safe_conversion_to_datetime).dt.date.apply(safe_conversion_to_datetime)
ct_timings_df = ct_timings_df.merge(registry_df[['SOS-CENTER-YEAR-NO.','Name', 'Date_birth', 'ct_date', 'DCI_ischemia',]], on=['SOS-CENTER-YEAR-NO.','Name', 'Date_birth', 'ct_date'], how='left')
ct_timings_df['DCI_ischemia'] = ct_timings_df['DCI_ischemia'].fillna(0).astype(int)
registry_df.drop(columns=['ct_date'], inplace=True)
ct_timings_df.drop(columns=['ct_date'], inplace=True)

Limit analysis to registry start

In [ ]:
registry_df['Date_admission'].min(), registry_df['Date_admission'].max()

In [ ]:
censored_registry_df = registry_df[registry_df['Date_admission'] >= '2008-01-01']
censored_ct_timings_df = ct_timings_df[ct_timings_df['Date_admission'] >= '2008-01-01']

In [ ]:
n_patients = censored_registry_df['Name'].nunique()
n_cts = censored_ct_timings_df.shape[0]

print('Number of patients in registry after 2019: {}'.format(n_patients))
print('Number of CTs after 2019: {}'.format(n_cts))

In [ ]:
n_dci_ischemia = censored_registry_df['DCI_ischemia'].sum()
n_dci_infarct = censored_registry_df['DCI_infarct'].sum()
n_cvs = censored_registry_df['CVS_YN'].sum()

print('Number of patients with DCI ischemia: {}'.format(n_dci_ischemia))
print('Number of patients with DCI infarct: {}'.format(n_dci_infarct))
print('Number of patients with CVS: {}'.format(n_cvs))

#### Compute timings

In [ ]:
import datetime

def registry_date_conversion(date):
    if isinstance(date, datetime.datetime):
        return date
    elif type(date) == str:
        return datetime.datetime.strptime(date, '%d.%m.%Y').date()
    else:
        return date



In [ ]:
# add Date_DCI_ischemia_first_image and Time_DCI_ischemia_first_image to get the full date
censored_registry_df['full_date_dci_ischemia'] = censored_registry_df['Date_DCI_ischemia_first_image'].apply(registry_date_conversion).astype(str) + ' ' + censored_registry_df['Time_DCI_ischemia_first_image'].astype(str)
# replace NaT nan with nan
censored_registry_df['full_date_dci_ischemia'] = censored_registry_df['full_date_dci_ischemia'].replace('NaT nan', pd.NaT)
censored_registry_df['full_date_dci_ischemia'] = censored_registry_df['full_date_dci_ischemia'].apply(safe_conversion_to_datetime)

censored_registry_df['full_date_dci_infarction'] = censored_registry_df['Date_DCI_infarct_first_image'].astype(str) + ' ' + censored_registry_df['Time_DCI_infarct_first_image'].astype(str)
# replace NaT nan with nan
censored_registry_df['full_date_dci_infarction'] = censored_registry_df['full_date_dci_infarction'].replace('NaT nan', pd.NaT)
censored_registry_df['full_date_dci_infarction'] = censored_registry_df['full_date_dci_infarction'].apply(safe_conversion_to_datetime)

# ensure number of nans in full_date_dci_ischemia and Date_DCI_ischemia_first_image are the same
assert censored_registry_df['full_date_dci_ischemia'].isnull().sum() == censored_registry_df['Date_DCI_ischemia_first_image'].isnull().sum()
# ensure number of nans in full_date_dci_infarction and Date_DCI_infarct_first_image are the same
assert censored_registry_df['full_date_dci_infarction'].isnull().sum() == censored_registry_df['Date_DCI_infarct_first_image'].isnull().sum()

In [ ]:
# compute time to CVS, DCI ischemia and DCI infarction
censored_registry_df['time_to_cvs'] = censored_registry_df['Date_CVS_Start'].apply(safe_conversion_to_datetime) - censored_registry_df['Date_Ictus'].apply(safe_conversion_to_datetime)

censored_registry_df['time_to_dci_ischemia'] = censored_registry_df['full_date_dci_ischemia'] - censored_registry_df['Date_Ictus'].apply(safe_conversion_to_datetime)
censored_registry_df['time_to_dci_infarction'] = censored_registry_df['full_date_dci_infarction'] - censored_registry_df['Date_Ictus'].apply(safe_conversion_to_datetime)

In [ ]:
# compute ct timings
censored_ct_timings_df['time_to_ct'] = censored_ct_timings_df['timeAktion'].apply(safe_conversion_to_datetime) - censored_ct_timings_df['Date_Ictus'].apply(safe_conversion_to_datetime)

In [ ]:
# check if any negative timings
print('Number of negative time_to_cvs: {}'.format((censored_registry_df['time_to_cvs'] < pd.Timedelta(0)).sum()))
print('Number of negative time_to_dci_ischemia: {}'.format((censored_registry_df['time_to_dci_ischemia'] < pd.Timedelta(0)).sum()))
print('Number of negative time_to_dci_infarction: {}'.format((censored_registry_df['time_to_dci_infarction'] < pd.Timedelta(0)).sum()))
print('Number of negative time_to_ct: {}'.format((censored_ct_timings_df['time_to_ct'] < pd.Timedelta(0)).sum()))

# filter out negative times
censored_registry_df.loc[censored_registry_df['time_to_cvs'] < pd.Timedelta(0), 'time_to_cvs'] = pd.NaT
censored_registry_df.loc[censored_registry_df['time_to_dci_ischemia'] < pd.Timedelta(0), 'time_to_dci_ischemia'] = pd.NaT
censored_registry_df.loc[censored_registry_df['time_to_dci_infarction'] < pd.Timedelta(0), 'time_to_dci_infarction'] = pd.NaT
censored_ct_timings_df.loc[censored_ct_timings_df['time_to_ct'] < pd.Timedelta(0), 'time_to_ct'] = pd.NaT

In [ ]:
restrict_ct_timings_to_dci = False
restrict_ct_timings_to_cvs = False
if restrict_ct_timings_to_cvs:
    censored_ct_timings_df = censored_ct_timings_df[censored_ct_timings_df['CVS_YN'] == 1]
if restrict_ct_timings_to_dci:
    censored_ct_timings_df = censored_ct_timings_df[censored_ct_timings_df['DCI_ischemia'] == 1]

## Evaluate time to CVS and DCI 

In [ ]:
censored_registry_df['time_to_cvs_days'] = censored_registry_df['time_to_cvs'].dt.total_seconds() / (60*60*24)
censored_registry_df['time_to_dci_ischemia_days'] = censored_registry_df['time_to_dci_ischemia'].dt.total_seconds() / (60*60*24)
censored_registry_df['time_to_dci_infarction_days'] = censored_registry_df['time_to_dci_infarction'].dt.total_seconds() / (60*60*24)
censored_ct_timings_df['time_to_ct_days'] = censored_ct_timings_df['time_to_ct'].dt.total_seconds() / (60*60*24)



In [ ]:
# n and percentage of patients with DCI ischemia after 10 days
days_limit = 21
print(f'Number of patients with DCI ischemia after {days_limit} days: {censored_registry_df[censored_registry_df["time_to_dci_ischemia_days"] > days_limit].shape[0]} ({censored_registry_df[censored_registry_df["time_to_dci_ischemia_days"] > days_limit].shape[0] / censored_registry_df["DCI_ischemia"].sum() * 100:.2f}%)')

# Base line data

In [ ]:
censored_registry_df.Fisher_Score = censored_registry_df.Fisher_Score.replace({'x': pd.NA, 'nan': pd.NA})
censored_registry_df.Fisher_Score = pd.to_numeric(censored_registry_df.Fisher_Score, errors='coerce')
censored_registry_df['Sex_encoded'] = censored_registry_df.Sex.apply(lambda x: 1 if x in ['M', 'm'] else 0)


In [ ]:
outcome_df.mRS_FU_1y = outcome_df.mRS_FU_1y.fillna(outcome_df.mRS_2FU_2y).fillna(outcome_df.mRS_3FU_5y)
# if mRS_discharge == 6, set mRS_FU_1y = 6
outcome_df.loc[outcome_df['mRS_discharge'] == 6, 'mRS_FU_1y'] = 6

outcome_df["mRS_FU_1y_int"] = pd.to_numeric(outcome_df["mRS_FU_1y"], errors="coerce")
outcome_df["mRS_discharge"] = pd.to_numeric(outcome_df["mRS_discharge"], errors="coerce")
outcome_df["Date_birth"] = pd.to_datetime(outcome_df["Date_birth"])

# Align death status to outcome rows using a shared patient identifier.
shared_id_col = next((col for col in ['SOS-CENTER-YEAR-NO.', 'pNr', 'Name'] if col in outcome_df.columns and col in censored_registry_df.columns), None)
if shared_id_col is not None:
    death_ids = set(censored_registry_df.loc[censored_registry_df['Death'] == 1, shared_id_col].dropna())
    outcome_df.loc[outcome_df[shared_id_col].isin(death_ids), "mRS_FU_1y_int"] = 6
else:
    print('Warning: no shared patient identifier between outcome_df and censored_registry_df; death override skipped.')

In [ ]:
def get_population_stats(registry_df, outcomes_df):
    population_df = pd.DataFrame()

    # Reference behavior from dci_timing.ipynb: use row count, not unique IDs.
    n_patients = registry_df.shape[0]
    population_df['n_patients'] = [n_patients]

    # Ensure aneurysm location indicator columns exist, following pupillometry logic.
    if 'Aneurysm_Artery_Code' in registry_df.columns:
        registry_df = registry_df.copy()
        registry_df['an_loc_acoma'] = registry_df['Aneurysm_Artery_Code'] == 8
        registry_df['an_loc_aca'] = registry_df['Aneurysm_Artery_Code'].isin([9, 22, 24])
        registry_df['an_loc_mca'] = registry_df['Aneurysm_Artery_Code'].isin([7, 20, 21])
        registry_df['an_loc_ica'] = registry_df['Aneurysm_Artery_Code'].isin([1, 2, 3, 4, 5, 6, 18, 19, 25, 26, 27, 29, 31])
        registry_df['an_loc_vert_bas_branches'] = registry_df['Aneurysm_Artery_Code'].isin([10, 11, 12, 13, 14, 15, 16, 17])
        registry_df['an_loc_pca'] = registry_df['Aneurysm_Artery_Code'].isin([23, 28])

    if 'los' not in registry_df.columns and {'Date_Discharge', 'Date_admission'}.issubset(registry_df.columns):
        registry_df = registry_df.copy()
        registry_df['los'] = (registry_df['Date_Discharge'].apply(safe_conversion_to_datetime) - registry_df['Date_admission'].apply(safe_conversion_to_datetime)).dt.days
    if 'los_icu' not in registry_df.columns and {'Date_discharge_ICU', 'Date_admission'}.issubset(registry_df.columns):
        registry_df = registry_df.copy()
        registry_df['los_icu'] = (registry_df['Date_discharge_ICU'].apply(safe_conversion_to_datetime) - registry_df['Date_admission'].apply(safe_conversion_to_datetime)).dt.days

    population_df['age_median'] = registry_df.Age.median()
    population_df['age_q1'] = registry_df.Age.quantile(0.25)
    population_df['age_q3'] = registry_df.Age.quantile(0.75)
    population_df['age_str'] = f'{population_df.age_median.iloc[0]:.1f} ({population_df.age_q1.iloc[0]:.1f}-{population_df.age_q3.iloc[0]:.1f})'

    sex_series = registry_df['Sex_encoded'] if 'Sex_encoded' in registry_df.columns else registry_df['Sex']
    population_df['n_female'] = sex_series.sum()
    population_df['p_female'] = sex_series.sum() / n_patients
    population_df['female_str'] = f'{population_df.n_female.iloc[0]} ({population_df.p_female.iloc[0]*100:.1f}%)'

    population_df['n_hta'] = registry_df.HTN.sum()
    population_df['p_hta'] = registry_df.HTN.sum() / n_patients
    population_df['hta_str'] = f'{population_df.n_hta.iloc[0]:.0f} ({population_df.p_hta.iloc[0]*100:.1f}%)'

    population_df['n_dm'] = registry_df.DM.sum()
    population_df['p_dm'] = registry_df.DM.sum() / n_patients
    population_df['dm_str'] = f'{population_df.n_dm.iloc[0]:.0f} ({population_df.p_dm.iloc[0]*100:.1f}%)'

    population_df['n_acoma'] = registry_df.an_loc_acoma.sum()
    population_df['p_acoma'] = registry_df.an_loc_acoma.sum() / n_patients
    population_df['acoma_str'] = f'{population_df.n_acoma.iloc[0]} ({population_df.p_acoma.iloc[0]*100:.1f}%)'

    population_df['n_aca'] = registry_df.an_loc_aca.sum()
    population_df['p_aca'] = registry_df.an_loc_aca.sum() / n_patients
    population_df['aca_str'] = f'{population_df.n_aca.iloc[0]} ({population_df.p_aca.iloc[0]*100:.1f}%)'

    population_df['n_mca'] = registry_df.an_loc_mca.sum()
    population_df['p_mca'] = registry_df.an_loc_mca.sum() / n_patients
    population_df['mca_str'] = f'{population_df.n_mca.iloc[0]} ({population_df.p_mca.iloc[0]*100:.1f}%)'

    population_df['n_pca'] = registry_df.an_loc_pca.sum()
    population_df['p_pca'] = registry_df.an_loc_pca.sum() / n_patients
    population_df['pca_str'] = f'{population_df.n_pca.iloc[0]} ({population_df.p_pca.iloc[0]*100:.1f}%)'

    population_df['n_ica'] = registry_df.an_loc_ica.sum()
    population_df['p_ica'] = registry_df.an_loc_ica.sum() / n_patients
    population_df['ica_str'] = f'{population_df.n_ica.iloc[0]} ({population_df.p_ica.iloc[0]*100:.1f}%)'

    population_df['n_vert_bas_branches'] = registry_df.an_loc_vert_bas_branches.sum()
    population_df['p_vert_bas_branches'] = registry_df.an_loc_vert_bas_branches.sum() / n_patients
    population_df['vert_bas_branches_str'] = f'{population_df.n_vert_bas_branches.iloc[0]} ({population_df.p_vert_bas_branches.iloc[0]*100:.1f}%)'

    population_df['loc_missing'] = registry_df.Aneurysm_Artery_Code.isna().sum()
    population_df['p_loc_missing'] = registry_df.Aneurysm_Artery_Code.isna().sum() / n_patients
    population_df['loc_missing_str'] = f'{population_df.loc_missing.iloc[0]} ({population_df.p_loc_missing.iloc[0]*100:.1f}%)'

    population_df['gcs_admission_median'] = registry_df.GCS_admission.median()
    population_df['gcs_admission_q1'] = registry_df.GCS_admission.quantile(0.25)
    population_df['gcs_admission_q3'] = registry_df.GCS_admission.quantile(0.75)
    population_df['gcs_admission_str'] = f'{population_df.gcs_admission_median.iloc[0]:.0f} ({population_df.gcs_admission_q1.iloc[0]:.0f}-{population_df.gcs_admission_q3.iloc[0]:.0f})'

    population_df['wfns_median'] = registry_df.WFNS.median()
    population_df['wfns_q1'] = registry_df.WFNS.quantile(0.25)
    population_df['wfns_q3'] = registry_df.WFNS.quantile(0.75)
    population_df['wfns_str'] = f'{population_df.wfns_median.iloc[0]:.0f} ({population_df.wfns_q1.iloc[0]:.0f}-{population_df.wfns_q3.iloc[0]:.0f})'

    population_df['fisher_median'] = pd.to_numeric(registry_df['Fisher_Score']).median()
    population_df['fisher_q1'] = pd.to_numeric(registry_df['Fisher_Score']).quantile(0.25)
    population_df['fisher_q3'] = pd.to_numeric(registry_df['Fisher_Score']).quantile(0.75)
    population_df['fisher_str'] = f'{population_df.fisher_median.iloc[0]:.0f} ({population_df.fisher_q1.iloc[0]:.0f}-{population_df.fisher_q3.iloc[0]:.0f})'

    # Coiling includes stenting as in the pupillometry table.
    population_df['n_coiling'] = (registry_df.Coiling + registry_df.Stenting).astype(bool).astype(int).sum()
    population_df['p_coiling'] = (registry_df.Coiling + registry_df.Stenting).astype(bool).astype(int).sum() / n_patients
    population_df['coiling_str'] = f'{population_df.n_coiling.iloc[0]:.0f} ({population_df.p_coiling.iloc[0]*100:.1f}%)'

    population_df['n_clipping'] = registry_df.Clipping.sum()
    population_df['p_clipping'] = registry_df.Clipping.sum() / n_patients
    population_df['clipping_str'] = f'{population_df.n_clipping.iloc[0]:.0f} ({population_df.p_clipping.iloc[0]*100:.1f}%)'

    population_df['los_icu_median'] = registry_df.los_icu.median()
    population_df['los_icu_q1'] = registry_df.los_icu.quantile(0.25)
    population_df['los_icu_q3'] = registry_df.los_icu.quantile(0.75)
    population_df['los_icu_str'] = f'{population_df.los_icu_median.iloc[0]:.0f} ({population_df.los_icu_q1.iloc[0]:.0f}-{population_df.los_icu_q3.iloc[0]:.0f})'

    population_df['los_median'] = registry_df.los.median()
    population_df['los_q1'] = registry_df.los.quantile(0.25)
    population_df['los_q3'] = registry_df.los.quantile(0.75)
    population_df['los_str'] = f'{population_df.los_median.iloc[0]:.0f} ({population_df.los_q1.iloc[0]:.0f}-{population_df.los_q3.iloc[0]:.0f})'

    population_df['n_mortality'] = registry_df.Death.sum()
    population_df['p_mortality'] = registry_df.Death.sum() / n_patients
    population_df['mortality_str'] = f'{population_df.n_mortality.iloc[0]:.0f} ({population_df.p_mortality.iloc[0]*100:.1f}%)'

    mrs_discharge_numeric = pd.to_numeric(outcomes_df['mRS_discharge'], errors='coerce')
    population_df['discharge_mrs_median'] = mrs_discharge_numeric.median()
    population_df['discharge_mrs_q1'] = mrs_discharge_numeric.quantile(0.25)
    population_df['discharge_mrs_q3'] = mrs_discharge_numeric.quantile(0.75)
    population_df['discharge_mrs_str'] = f'{population_df["discharge_mrs_median"].iloc[0]:.0f} ({population_df["discharge_mrs_q1"].iloc[0]:.0f}-{population_df["discharge_mrs_q3"].iloc[0]:.0f})'

    mrs_numeric = pd.to_numeric(outcomes_df['mRS_FU_1y'], errors='coerce')
    population_df['1y_mrs_median'] = mrs_numeric.median()
    population_df['1y_mrs_q1'] = mrs_numeric.quantile(0.25)
    population_df['1y_mrs_q3'] = mrs_numeric.quantile(0.75)
    population_df['1y_mrs_str'] = f'{population_df["1y_mrs_median"].iloc[0]:.0f} ({population_df["1y_mrs_q1"].iloc[0]:.0f}-{population_df["1y_mrs_q3"].iloc[0]:.0f})'

    str_cols = ['n_patients'] + [col for col in population_df.columns if col.endswith('_str')]
    str_df = population_df[str_cols].copy()
    str_df['n_patients'] = str_df['n_patients'].astype(int).astype(str)

    return population_df, str_df

In [ ]:
patient_id_col = 'SOS-CENTER-YEAR-NO.' if 'SOS-CENTER-YEAR-NO.' in outcome_df.columns else 'Name'

all_registry_df = censored_registry_df
all_outcome_df = outcome_df[outcome_df[patient_id_col].isin(all_registry_df[patient_id_col])]

dci_registry_df = censored_registry_df[censored_registry_df['DCI_ischemia'] == 1]
dci_outcome_df = outcome_df[outcome_df[patient_id_col].isin(dci_registry_df[patient_id_col])]

no_dci_registry_df = censored_registry_df[censored_registry_df['DCI_ischemia'] == 0]
no_dci_outcome_df = outcome_df[outcome_df[patient_id_col].isin(no_dci_registry_df[patient_id_col])]

all_pop_df, all_str_df = get_population_stats(all_registry_df, all_outcome_df)
dci_pop_df, dci_str_df = get_population_stats(dci_registry_df, dci_outcome_df)
no_dci_pop_df, no_dci_str_df = get_population_stats(no_dci_registry_df, no_dci_outcome_df)

joined_pop_df = pd.concat([all_pop_df.T, dci_pop_df.T, no_dci_pop_df.T], axis=1)
joined_pop_df.columns = ['All patients', 'DCI ischemia patients', 'No DCI']

joined_str_pop_df = pd.concat([all_str_df.T, dci_str_df.T, no_dci_str_df.T], axis=1)
joined_str_pop_df.columns = ['All patients', 'DCI ischemia patients', 'No DCI']

In [ ]:
joined_str_pop_df

In [ ]:
# joined_pop_df.to_csv('/Users/jk1/Downloads/dci_timing_table1.csv')

## Formatted table

In [ ]:
n_all = int(joined_pop_df.loc['n_patients', 'All patients'])
n_dci = int(joined_pop_df.loc['n_patients', 'DCI ischemia patients'])
n_no_dci = int(joined_pop_df.loc['n_patients', 'No DCI'])

col_all = f'Overall Population\n(n = {n_all})'
col_dci = f'DCI\n(n = {n_dci})'
col_no_dci = f'No DCI\n(n = {n_no_dci})'

def _safe_str(idx, col):
    return joined_str_pop_df.loc[idx, col] if idx in joined_str_pop_df.index else 'NA'

def _fmt_count_pct(df, col):
    if col not in df.columns:
        return 'NA'
    n = int(df[col].sum())
    d = int(df.shape[0])
    p = (100.0 * n / d) if d else 0.0
    return f'{n} ({p:.1f}%)'

rows = [
    ['Demographics', '', '', ''],
    ['  Age', _safe_str('age_str', 'All patients'), _safe_str('age_str', 'DCI ischemia patients'), _safe_str('age_str', 'No DCI')],
    ['  Sex (female)', _safe_str('female_str', 'All patients'), _safe_str('female_str', 'DCI ischemia patients'), _safe_str('female_str', 'No DCI')],
    ['Risk factors', '', '', ''],
    ['  Hypertension', _safe_str('hta_str', 'All patients'), _safe_str('hta_str', 'DCI ischemia patients'), _safe_str('hta_str', 'No DCI')],
    ['  Diabetes', _safe_str('dm_str', 'All patients'), _safe_str('dm_str', 'DCI ischemia patients'), _safe_str('dm_str', 'No DCI')],
    ['Aneurysm location', '', '', ''],
    ['  Anterior communicating artery', _safe_str('acoma_str', 'All patients'), _safe_str('acoma_str', 'DCI ischemia patients'), _safe_str('acoma_str', 'No DCI')],
    ['  Anterior cerebral artery', _safe_str('aca_str', 'All patients'), _safe_str('aca_str', 'DCI ischemia patients'), _safe_str('aca_str', 'No DCI')],
    ['  Middle cerebral artery', _safe_str('mca_str', 'All patients'), _safe_str('mca_str', 'DCI ischemia patients'), _safe_str('mca_str', 'No DCI')],
    ['  Posterior cerebral artery', _safe_str('pca_str', 'All patients'), _safe_str('pca_str', 'DCI ischemia patients'), _safe_str('pca_str', 'No DCI')],
    ['  Internal carotid artery', _safe_str('ica_str', 'All patients'), _safe_str('ica_str', 'DCI ischemia patients'), _safe_str('ica_str', 'No DCI')],
    ['  Vertebral/basilar artery', _safe_str('vert_bas_branches_str', 'All patients'), _safe_str('vert_bas_branches_str', 'DCI ischemia patients'), _safe_str('vert_bas_branches_str', 'No DCI')],
    ['Admission status', '', '', ''],
    ['  Glasgow Coma Scale', _safe_str('gcs_admission_str', 'All patients'), _safe_str('gcs_admission_str', 'DCI ischemia patients'), _safe_str('gcs_admission_str', 'No DCI')],
    ['  World Federation of Neurological Surgeons Scale', _safe_str('wfns_str', 'All patients'), _safe_str('wfns_str', 'DCI ischemia patients'), _safe_str('wfns_str', 'No DCI')],
    ['  Modified Fisher Scale', _safe_str('fisher_str', 'All patients'), _safe_str('fisher_str', 'DCI ischemia patients'), _safe_str('fisher_str', 'No DCI')],
    ['Acute treatment', '', '', ''],
    ['  Coiling', _safe_str('coiling_str', 'All patients'), _safe_str('coiling_str', 'DCI ischemia patients'), _safe_str('coiling_str', 'No DCI')],
    ['  Clipping', _safe_str('clipping_str', 'All patients'), _safe_str('clipping_str', 'DCI ischemia patients'), _safe_str('clipping_str', 'No DCI')],
    ['Outcomes', '', '', ''],
    ['  Vasospasm', _fmt_count_pct(all_registry_df, 'CVS_YN'), _fmt_count_pct(dci_registry_df, 'CVS_YN'), _fmt_count_pct(no_dci_registry_df, 'CVS_YN')],
    ['  ICU length of stay (d)', _safe_str('los_icu_str', 'All patients'), _safe_str('los_icu_str', 'DCI ischemia patients'), _safe_str('los_icu_str', 'No DCI')],
    ['  Hospital length of stay (d)', _safe_str('los_str', 'All patients'), _safe_str('los_str', 'DCI ischemia patients'), _safe_str('los_str', 'No DCI')],
    ['  Hospital mortality', _safe_str('mortality_str', 'All patients'), _safe_str('mortality_str', 'DCI ischemia patients'), _safe_str('mortality_str', 'No DCI')],
    ['  Discharge modified Rankin Scale', _safe_str('discharge_mrs_str', 'All patients'), _safe_str('discharge_mrs_str', 'DCI ischemia patients'), _safe_str('discharge_mrs_str', 'No DCI')],
    ['  1-yr modified Rankin Scale', _safe_str('1y_mrs_str', 'All patients'), _safe_str('1y_mrs_str', 'DCI ischemia patients'), _safe_str('1y_mrs_str', 'No DCI')],
]

final_table_df = pd.DataFrame(rows, columns=['Variable', col_all, col_dci, col_no_dci])
section_names = {'Demographics', 'Risk factors', 'Aneurysm location', 'Admission status', 'Acute treatment', 'Outcomes'}

styled_final_table = (
    final_table_df.style
    .hide(axis='index')
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#57595d'), ('color', 'white'), ('font-weight', 'bold')]},
        {'selector': 'td', 'props': [('background-color', '#d3d4d6'), ('color', '#222'), ('border', 'none')]},
        {'selector': 'td.col0', 'props': [('text-align', 'left')]},
        {'selector': 'td:not(.col0)', 'props': [('text-align', 'center')]},
    ])
    .apply(lambda row: ['background-color: #c3c5c8; font-weight: bold;' if row['Variable'].strip() in section_names else '' for _ in row], axis=1)
    .format(na_rep='NA')
)

styled_final_table